# Theorem 11 — private-preserving optimal transport

**Formal source:** [`../11_private_preserving_optimal_transport.md`](../11_private_preserving_optimal_transport.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
source = np.array([[-1, -2], [-1, 2], [1, -2], [1, 2]], float)
target = np.array([[0, -2], [0, 2], [2, -2], [2, 2]], float)
lam = 3

def transport_cost(left, right):
    return (left[0] - right[0]) ** 2 + lam * (left[1] - right[1]) ** 2

candidates = []
for permutation in itertools.permutations(range(4)):
    candidates.append((sum(transport_cost(source[i], target[permutation[i]]) for i in range(4)) / 4, permutation))
best_cost, best_permutation = min(candidates, key=lambda item: item[0])
private_drift = np.mean([abs(source[i, 1] - target[best_permutation[i], 1]) for i in range(4)])
assert best_cost == 1 and private_drift == 0
print({"optimal_cost": float(best_cost), "private_drift": float(private_drift)})

In [ ]:
print('THEORY_DEMO_PASS::11_private_preserving_optimal_transport')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')